In [1]:
import requests
# import bs4
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

In [2]:
BASE_URL = "https://stockanalysis.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [5]:
# res = requests.get(BASE_URL + "/stocks/industry/")
# res.raise_for_status()
def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")



In [6]:
soup = get_soup(BASE_URL + "/stocks/industry/")


In [9]:
def get_subsector_urls():
    counter = 0
    arr_sectors = []
    sector_tables = soup.select(".styled-scrollbar > table")
    # print (len(sector_tables))
    for t in soup.select(".controls > .title-group"):
        if counter <= 100:
            href = t.select('a')[0]['href']
            sector_key = href.replace("/stocks/sector/", "").replace("/", "")
            
            sector_table = sector_tables[counter]
            subsector_keys = [str(x['href']).replace("/stocks/industry/", "").replace("/", "") for x in sector_table.select("a")]
            subsector_hrefs = [x['href'] for x in sector_table.select("a")]

            arr_sectors.append({"sector_name": sector_key, "subsector_keys": subsector_keys,  "urls": subsector_hrefs})
          
        counter += 1

    return arr_sectors

arr_sectors = get_subsector_urls()
g_sub_sector_soup = None
arr_sectors

[{'sector_name': 'healthcare',
  'subsector_keys': ['biotechnology',
   'medical-devices',
   'drug-manufacturers-specialty-and-generic',
   'medical-instruments-and-supplies',
   'medical-care-facilities',
   'diagnostics-and-research',
   'health-information-services',
   'drug-manufacturers-general',
   'medical-distribution',
   'healthcare-plans',
   'pharmaceutical-retailers'],
  'urls': ['/stocks/industry/biotechnology/',
   '/stocks/industry/medical-devices/',
   '/stocks/industry/drug-manufacturers-specialty-and-generic/',
   '/stocks/industry/medical-instruments-and-supplies/',
   '/stocks/industry/medical-care-facilities/',
   '/stocks/industry/diagnostics-and-research/',
   '/stocks/industry/health-information-services/',
   '/stocks/industry/drug-manufacturers-general/',
   '/stocks/industry/medical-distribution/',
   '/stocks/industry/healthcare-plans/',
   '/stocks/industry/pharmaceutical-retailers/']},
 {'sector_name': 'financials',
  'subsector_keys': ['banks-regional'

,ticker,company_name,sub_sector_name


In [10]:
def scrape_sectors(sectors):
    counter2 = 0
    df_all_sectors = pd.DataFrame(columns=['ticker', 'company_name', 'sector_name', 'sub_sector_name']) 
    for sector in arr_sectors:
        sector_name = sector.get("sector_name")

        # 'energy', 
        if sector_name not in ['healthcare','financials', 'technology', 'industrials', 'communication-services', 'consumer-staples', 'materials', 'consumer-discretionary', 'real-estate']:
         
            if counter2 <= 0:        
                df_sector = pd.DataFrame(columns=['ticker', 'company_name', 'sector_name','sub_sector_name'])
                
                print ('Getting stocks for ', sector.get("sector_name"))
                    
                for url in sector.get("urls"):
                    subsector_name = url[0: len(url) - 1]
                    subsector_name = subsector_name[subsector_name.rindex("/") + 1:]
                    df = scrape_page_for_tickers(BASE_URL +  url, sector_name)
                    df_sector = pd.concat([df_sector, df], ignore_index=True)
                    time.sleep(2)
            
            
                df_all_sectors = pd.concat([df_all_sectors, df_sector], ignore_index=True)
            
            counter2 += 1
        
    return df_all_sectors

In [11]:
def scrape_page_for_tickers(url, sector_name):
    global g_sub_sector_soup
    
    df = None
    subsector_name = None
    
    if g_sub_sector_soup is None:
        g_sub_sector_soup = get_soup(url)
        print('getting soup')

    soup = get_soup(url)
    
    subsector_name = url
    subsector_name = subsector_name[0: len(subsector_name) - 1]
    subsector_name = subsector_name[subsector_name.rindex("/") + 1:]
    total_pages = get_total_pages(soup)
    
    all_rows = []
    
    print(f"{subsector_name}: {total_pages} pages found. url: {url}")
    
    # scrape page 1
    all_rows.extend(scrape_page(url, sector_name, subsector_name))
    print(f"Scraping page 1 of url: {url}")

    # # scrape remaining pages
    for page_num in range(2, total_pages + 1):
        page_url = f"{url}?page={page_num}"
        print(f"Scraping page {page_num}... url: {page_url}")
    
        rows = scrape_page(page_url, sector_name, subsector_name)
        all_rows.extend(rows)
    
    df = pd.DataFrame(all_rows)
    
    # # remove duplicates if any
    df.drop_duplicates(subset=["ticker"], inplace=True)
    # # save csv
    # file_name = f"{subsector_name}.csv"
    # df.to_csv(file_name, index=False)
    # print(f"Saved {len(df)} stocks to {file_name}")
    return df


    

In [12]:
def scrape_page(url, sector_name, sub_sector_name):
    soup = get_soup(url)
    # print (url)

    rows = []

    ticker_cells = soup.find_all("td", class_="sym")

    for ticker_cell in ticker_cells:

        a = ticker_cell.find("a")
        if not a:
            continue

        ticker = a.get_text(strip=True)

        company_cell = ticker_cell.find_next_sibling("td", class_="slw")
        company_name = company_cell.get_text(strip=True) if company_cell else ""

        rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "sector_name": sector_name,
            "sub_sector_name": sub_sector_name
        })

    return rows
    
def get_total_pages(soup):
    
    nav = soup.find_all("nav")
    
    if len(nav) == 2:
        return 1
        
    if not nav:
        return 1

    nav = nav[2]
    
    # Find the outer span that contains pagination text
    page_span = nav.find("span", class_="whitespace-nowrap")

    if not page_span:
        return 1

    text = page_span.get_text(" ", strip=True)
    # Example result: "Page 1 of 2"

    match = re.search(r"of\s+(\d+)", text)

    if match:
        return int(match.group(1))

    return 1

In [13]:
    
df_all = scrape_sectors(arr_sectors)
df_all.to_csv("stocks_all_sectors_energy.csv", index=False)


Getting stocks for  energy
getting soup
oil-gas-e-and-p: 1 pages found. url: https://stockanalysis.com/stocks/industry/oil-gas-e-and-p/
Scraping page 1 of url: https://stockanalysis.com/stocks/industry/oil-gas-e-and-p/
oil-gas-midstream: 1 pages found. url: https://stockanalysis.com/stocks/industry/oil-gas-midstream/
Scraping page 1 of url: https://stockanalysis.com/stocks/industry/oil-gas-midstream/
oil-gas-equipment-and-services: 1 pages found. url: https://stockanalysis.com/stocks/industry/oil-gas-equipment-and-services/
Scraping page 1 of url: https://stockanalysis.com/stocks/industry/oil-gas-equipment-and-services/
oil-gas-integrated: 1 pages found. url: https://stockanalysis.com/stocks/industry/oil-gas-integrated/
Scraping page 1 of url: https://stockanalysis.com/stocks/industry/oil-gas-integrated/
oil-gas-refining-and-marketing: 1 pages found. url: https://stockanalysis.com/stocks/industry/oil-gas-refining-and-marketing/
Scraping page 1 of url: https://stockanalysis.com/stocks/i

In [21]:
table_1 = soup.select(".styled-scrollbar > table")[1]
table_1.select("tr:not(:first-child)")

[<tr class="svelte-1vx0bpj"><!--[--><!--[--><td class="svelte-1vx0bpj"><!-- --><a href="/stocks/industry/shell-companies/">Shell Companies</a><!-- --></td><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- -->291<!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- -->76.26B<!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- -->-<!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- -->-<!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- -->-232.03%<!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- --><span class="rr">-0.34%</span><!-- --></td><!--]--><!--]--><!--]--><!--[!--><!--[--><!--[--><td class="tr svelte-1vx0bpj"><!-- --><span class="rr">-1.40%</span><!-- --></td><!--]--><!--]--><!--]--><!--]--></tr>,
 <tr class="svelte-1vx0bpj"><!--[--><!--[--